# HMM Training Workflow — Modification Detection

This notebook demonstrates the **HMM training modes** for RNA modification
detection using the baleen eventalign pipeline. We show three training approaches:

- **Mode B: Semi-supervised** — Platt-scaling calibration of V2 mixture probabilities
- **Mode C: Fully supervised** — MLE transitions + KDE emission densities
- **Cross-validation** — Leave-one-contig-out validation to assess generalization

Training is performed on *E. coli* rRNA with known modification sites, and
parameters can be exported for transfer to other species (yeast, human, etc.).

## 0. Setup

In [ ]:
%matplotlib inline

import logging
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Enable INFO-level logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)

print("Environment ready.")

## Section A: Prepare labels from known modifications

Convert experimentally validated biological modification sites to training
labels for the HMM. The `labels_from_known_modifications()` helper applies
the **+3 position offset** (eventalign reports kmer start, modification is
at centre) automatically.

In [ ]:
from collections import Counter

# Known E. coli rRNA modifications — biological positions (1-based)
# Format: {(contig, biological_position): (short_name, full_name)}
KNOWN_MODIFICATIONS = {
    # --- Pseudouridine (Psi) ---
    ("ecoli16S", 516):  ("Psi", "pseudouridine"),
    ("ecoli23S", 746):  ("Psi", "pseudouridine"),
    ("ecoli23S", 955):  ("Psi", "pseudouridine"),
    ("ecoli23S", 1911): ("Psi", "pseudouridine"),
    ("ecoli23S", 1917): ("Psi", "pseudouridine"),
    ("ecoli23S", 2457): ("Psi", "pseudouridine"),
    ("ecoli23S", 2504): ("Psi", "pseudouridine"),
    ("ecoli23S", 2580): ("Psi", "pseudouridine"),
    ("ecoli23S", 2604): ("Psi", "pseudouridine"),
    ("ecoli23S", 2605): ("Psi", "pseudouridine"),
    # --- N2-methylguanosine (m2G) ---
    ("ecoli16S", 966):  ("m2G", "N2-methylguanosine"),
    ("ecoli16S", 1207): ("m2G", "N2-methylguanosine"),
    ("ecoli16S", 1516): ("m2G", "N2-methylguanosine"),
    ("ecoli23S", 1835): ("m2G", "N2-methylguanosine"),
    ("ecoli23S", 2445): ("m2G", "N2-methylguanosine"),
    # --- 5-methylcytidine (m5C) ---
    ("ecoli16S", 967):  ("m5C", "5-methylcytidine"),
    ("ecoli16S", 1407): ("m5C", "5-methylcytidine"),
    ("ecoli23S", 1962): ("m5C", "5-methylcytidine"),
    # --- 5-methyluridine (m5U) ---
    ("ecoli23S", 747):  ("m5U", "5-methyluridine"),
    ("ecoli23S", 1939): ("m5U", "5-methyluridine"),
    # --- N6,N6-dimethyladenosine (m66A) ---
    ("ecoli16S", 1518): ("m66A", "N6,N6-dimethyladenosine"),
    ("ecoli16S", 1519): ("m66A", "N6,N6-dimethyladenosine"),
    # --- N6-methyladenosine (m6A) ---
    ("ecoli23S", 1618): ("m6A", "N6-methyladenosine"),
    ("ecoli23S", 2030): ("m6A", "N6-methyladenosine"),
    # --- N7-methylguanosine (m7G) ---
    ("ecoli16S", 527):  ("m7G", "N7-methylguanosine"),
    ("ecoli23S", 2069): ("m7G", "N7-methylguanosine"),
    # --- Single-occurrence modifications ---
    ("ecoli23S", 2498): ("Cm",    "2′-O-methyl cytosine"),
    ("ecoli23S", 2449): ("D",     "dihydrouridine"),
    ("ecoli23S", 2251): ("Gm",    "2′-O-methyl guanine"),
    ("ecoli23S", 2552): ("Um",    "2′-O-methyl uridine"),
    ("ecoli23S", 2501): ("ho5C",  "5-hydroxycytidine"),
    ("ecoli23S", 745):  ("m1G",   "1-methylguanosine"),
    ("ecoli23S", 2503): ("m2A",   "2-methyladenosine"),
    ("ecoli23S", 1915): ("m3Psi", "3-methylpseudouridine"),
    ("ecoli16S", 1498): ("m3U",   "3-methyluridine"),
    ("ecoli16S", 1402): ("m4Cm",  "N4,2′-O-dimethylcytidine"),
}

mod_counts = Counter(v[0] for v in KNOWN_MODIFICATIONS.values())
print(f"Total known modification sites: {len(KNOWN_MODIFICATIONS)}")
print(f"Modification types: {len(mod_counts)}\n")
for mod_type, count in mod_counts.most_common():
    print(f"  {mod_type:<8s} {count:3d}")

In [ ]:
from baleen.eventalign import load_results, labels_from_known_modifications

# Load pipeline results (assumes you've already run the eventalign pipeline)
# Replace with your actual output path
RESULTS_PATH = Path("../output/eventalign_test/pipeline_results.pkl")

if not RESULTS_PATH.exists():
    print(f"ERROR: Pipeline results not found at {RESULTS_PATH}")
    print("Please run the eventalign_pipeline.ipynb notebook first.")
else:
    contig_results, metadata = load_results(RESULTS_PATH)
    print(f"Loaded results for {len(contig_results)} contigs")
    
    # Convert known mods to labels
    # The offset=3 is applied automatically inside labels_from_known_modifications
    labels = labels_from_known_modifications(
        KNOWN_MODIFICATIONS,
        contig_results,
        position_offset=3,
    )
    
    print(f"\nGenerated labels:")
    for contig, label_dict in sorted(labels.items()):
        n_mod = sum(1 for v in label_dict.values() if v == 1)
        n_unmod = len(label_dict) - n_mod
        print(f"  {contig:15s}: {len(label_dict):4d} positions "
              f"({n_mod} modified, {n_unmod} unmodified)")

## Section B: Run V1→V2 on all contigs

Compute the hierarchical pipeline (empirical-Bayes null + anchored mixture)
**without HMM smoothing** to get the raw V2 probabilities. These serve as
emission observations for training.

In [ ]:
from baleen.eventalign import compute_sequential_modification_probabilities

# Store V2 results per contig
v2_results = {}

for contig, cr in sorted(contig_results.items()):
    print(f"Running V1→V2 (no HMM) on {contig} ({len(cr.positions)} positions)...")
    hres = compute_sequential_modification_probabilities(cr, run_hmm=False)
    v2_results[contig] = hres
    print(f"  → {len(hres.position_stats)} positions with mixture results")

print(f"\nV2 results ready for {len(v2_results)} contigs.")

## Section C: Mode B — Semi-supervised training (Platt scaling)

**Semi-supervised mode** fits a logistic (Platt-scaling) calibrator to the
V2 mixture probabilities using known modification labels. The calibrator
parameters (a, b) define the emission log-odds transformation:

$$
\log \frac{P(x | \text{modified})}{P(x | \text{unmodified})} = a \cdot \text{logit}(p_{\text{raw}}) + b
$$

Transition probabilities are set to a **default p_stay=0.9999**.

In [ ]:
from baleen.eventalign import train_semi_supervised

# Train Mode B
params_b = train_semi_supervised(
    v2_results,
    labels,
    species_name="E. coli rRNA",
)

print(f"\nMode B — Semi-supervised (Platt scaling)")
print(f"  Species: {params_b.species_name}")
print(f"  Calibrator: a={params_b.emission_model.a:.4f}, b={params_b.emission_model.b:.4f}")
print(f"  Init prob (modified): {params_b.init_prob_modified:.6f}")
print(f"  Transition p(stay): {params_b.p_stay:.6f}")

In [ ]:
# Visualize calibration curve
p_raw = np.linspace(0.01, 0.99, 200)
logit_raw = np.log(p_raw / (1 - p_raw))
a, b = params_b.emission_model.a, params_b.emission_model.b
log_odds = a * logit_raw + b
p_calibrated = 1 / (1 + np.exp(-log_odds))

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(p_raw, p_calibrated, linewidth=2, label="Calibrator")
ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1, label="Identity")
ax.set_xlabel("Raw V2 P(modified)")
ax.set_ylabel("Calibrated P(modified)")
ax.set_title(f"Mode B Calibration Curve\na={a:.4f}, b={b:.4f}", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## Section D: Mode C — Fully supervised training (MLE + KDE)

**Fully supervised mode** learns both transitions and emissions from labeled data:

- **Transitions**: MLE from consecutive position pairs → per-base p_stay
- **Emissions**: KDE on raw V2 probabilities for modified vs unmodified

In [ ]:
from baleen.eventalign import train_supervised

# Train Mode C
params_c = train_supervised(
    v2_results,
    labels,
    species_name="E. coli rRNA",
    kde_n_bins=200,
)

print(f"\nMode C — Fully supervised (MLE + KDE)")
print(f"  Species: {params_c.species_name}")
print(f"  Init prob (modified): {params_c.init_prob_modified:.6f}")
print(f"  Learned p(stay) per base: {params_c.p_stay:.6f}")
print(f"  KDE bins: {len(params_c.emission_model.unmod_density)}")

In [ ]:
# Plot KDE emission densities
fig, ax = plt.subplots(figsize=(8, 5))

x = params_c.emission_model.bin_centers
unmod_dens = params_c.emission_model.unmod_density
mod_dens = params_c.emission_model.mod_density

ax.plot(x, unmod_dens, linewidth=2, label="Unmodified", color="#70AD47")
ax.plot(x, mod_dens, linewidth=2, label="Modified", color="#C44E52")
ax.fill_between(x, 0, unmod_dens, alpha=0.3, color="#70AD47")
ax.fill_between(x, 0, mod_dens, alpha=0.3, color="#C44E52")

ax.set_xlabel("Raw V2 P(modified)")
ax.set_ylabel("Density")
ax.set_title(f"Mode C Emission Densities (KDE)\np_stay={params_c.p_stay:.6f}", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

## Section E: Cross-validation

Perform **leave-one-contig-out cross-validation** to estimate generalization
performance. For each fold, we train on N-1 contigs and test on the held-out
contig, computing AUROC and AUPRC.

In [ ]:
from baleen.eventalign import cross_validate_hmm

# Cross-validate Mode B (semi-supervised)
print("Running leave-one-contig-out CV for Mode B (semi-supervised)...")
cv_results_b = cross_validate_hmm(
    v2_results,
    labels,
    mode="semi_supervised",
    cv_strategy="leave_one_contig_out",
)

print(f"\nMode B CV Results ({len(cv_results_b.fold_results)} folds):")
for fold in cv_results_b.fold_results:
    print(f"  Fold (test={fold.test_contig}): "
          f"AUROC={fold.auroc:.4f}, AUPRC={fold.auprc:.4f}")
print(f"  Mean AUROC: {cv_results_b.mean_auroc:.4f} ± {cv_results_b.std_auroc:.4f}")
print(f"  Mean AUPRC: {cv_results_b.mean_auprc:.4f} ± {cv_results_b.std_auprc:.4f}")

In [ ]:
# Cross-validate Mode C (fully supervised)
print("\nRunning leave-one-contig-out CV for Mode C (fully supervised)...")
cv_results_c = cross_validate_hmm(
    v2_results,
    labels,
    mode="supervised",
    cv_strategy="leave_one_contig_out",
)

print(f"\nMode C CV Results ({len(cv_results_c.fold_results)} folds):")
for fold in cv_results_c.fold_results:
    print(f"  Fold (test={fold.test_contig}): "
          f"AUROC={fold.auroc:.4f}, AUPRC={fold.auprc:.4f}")
print(f"  Mean AUROC: {cv_results_c.mean_auroc:.4f} ± {cv_results_c.std_auroc:.4f}")
print(f"  Mean AUPRC: {cv_results_c.mean_auprc:.4f} ± {cv_results_c.std_auprc:.4f}")

In [ ]:
# Comparison table
import pandas as pd

comparison = pd.DataFrame([
    {
        "Mode": "B (semi-supervised)",
        "Mean AUROC": f"{cv_results_b.mean_auroc:.4f}",
        "Std AUROC": f"{cv_results_b.std_auroc:.4f}",
        "Mean AUPRC": f"{cv_results_b.mean_auprc:.4f}",
        "Std AUPRC": f"{cv_results_b.std_auprc:.4f}",
        "N folds": len(cv_results_b.fold_results),
    },
    {
        "Mode": "C (fully supervised)",
        "Mean AUROC": f"{cv_results_c.mean_auroc:.4f}",
        "Std AUROC": f"{cv_results_c.std_auroc:.4f}",
        "Mean AUPRC": f"{cv_results_c.mean_auprc:.4f}",
        "Std AUPRC": f"{cv_results_c.std_auprc:.4f}",
        "N folds": len(cv_results_c.fold_results),
    },
])

print("\n=== Cross-Validation Comparison ===")
print(comparison.to_string(index=False))

## Section F: Export trained parameters

Save the learned HMM parameters to JSON files for:

1. **Cross-species transfer** — apply E. coli params to yeast/human data
2. **Reproducibility** — freeze params for production pipelines
3. **Version control** — track parameter evolution over time

In [ ]:
from baleen.eventalign import save_hmm_params, load_hmm_params

# Create output directory
PARAMS_DIR = Path("../output/hmm_params")
PARAMS_DIR.mkdir(parents=True, exist_ok=True)

# Save Mode B params
path_b = PARAMS_DIR / "ecoli_semi_supervised.json"
save_hmm_params(params_b, path_b)
print(f"Saved Mode B params to: {path_b}")

# Save Mode C params
path_c = PARAMS_DIR / "ecoli_supervised.json"
save_hmm_params(params_c, path_c)
print(f"Saved Mode C params to: {path_c}")

# Demonstrate loading
loaded_params = load_hmm_params(path_b)
print(f"\nLoaded params: {loaded_params.species_name} (mode={loaded_params.mode})")
print(f"  Emission type: {type(loaded_params.emission_model).__name__}")
print(f"  p_stay: {loaded_params.p_stay:.6f}")

## Section G: Placeholder for other species

To train HMMs for **yeast** or **human** data:

1. Run the eventalign pipeline on your native + IVT samples
2. Define known modification sites for your species
3. Generate labels using `labels_from_known_modifications()`
4. Train with the same workflow (Sections A–F)

You can also **transfer** E. coli params to a new species by loading the
JSON and passing `hmm_params=loaded_params` to `compute_sequential_modification_probabilities()`.

In [ ]:
# Example: Load yeast pipeline results (placeholder)
# YEAST_RESULTS_PATH = Path("../output/yeast/pipeline_results.pkl")
# if YEAST_RESULTS_PATH.exists():
#     yeast_contig_results, yeast_metadata = load_results(YEAST_RESULTS_PATH)
#     print(f"Loaded yeast results for {len(yeast_contig_results)} contigs")
# else:
#     print(f"Yeast data not found at {YEAST_RESULTS_PATH}")
#     print("Run the eventalign pipeline on yeast samples first.")

print("Placeholder cell for yeast/human data workflow.")
print("See comments above for example code.")

In [ ]:
# Example: Define known yeast modifications (placeholder)
# KNOWN_YEAST_MODS = {
#     ("yeast18S", 100): ("Psi", "pseudouridine"),
#     # ... add more positions
# }
#
# yeast_labels = labels_from_known_modifications(
#     KNOWN_YEAST_MODS,
#     yeast_contig_results,
#     position_offset=3,
# )
#
# # Train on yeast data
# yeast_v2_results = {}
# for contig, cr in yeast_contig_results.items():
#     hres = compute_sequential_modification_probabilities(cr, run_hmm=False)
#     yeast_v2_results[contig] = hres
#
# yeast_params = train_semi_supervised(yeast_v2_results, yeast_labels, species_name="Yeast")
# save_hmm_params(yeast_params, PARAMS_DIR / "yeast_semi_supervised.json")

print("Placeholder cell for yeast training workflow.")
print("See comments above for example code.")

In [ ]:
# Example: Apply E. coli params to yeast data (transfer learning)
# ecoli_params = load_hmm_params(PARAMS_DIR / "ecoli_semi_supervised.json")
#
# for contig, cr in yeast_contig_results.items():
#     hres = compute_sequential_modification_probabilities(
#         cr,
#         run_hmm=True,
#         hmm_params=ecoli_params,
#     )
#     # Now hres.position_stats[pos].p_mod_hmm contains HMM-smoothed probabilities
#     # using E. coli-trained parameters

print("Placeholder cell for transfer learning workflow.")
print("See comments above for example code.")

## Summary

This notebook demonstrated:

✅ **Section A**: Label preparation from known modification sites  
✅ **Section B**: V1→V2 pipeline (no HMM) to get raw probabilities  
✅ **Section C**: Mode B semi-supervised training (Platt scaling)  
✅ **Section D**: Mode C fully supervised training (MLE + KDE)  
✅ **Section E**: Cross-validation to assess generalization  
✅ **Section F**: Export/import trained parameters  
✅ **Section G**: Workflow templates for other species  

**Next steps**:

1. Run the full pipeline on your own data with `eventalign_pipeline.ipynb`
2. Adapt this notebook to your species and known modification sites
3. Compare Mode B vs Mode C performance on your data
4. Use the exported parameters in production pipelines